#  第四章 LangChain应用级系统设计与RAG实践

#### 4.1.2.2 基础实践：单输入输出线性流转

In [1]:
# =========================
# 1. 基础依赖
# =========================
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from dotenv import load_dotenv
import os

# =========================
# 2. 环境变量 & 模型
# =========================
load_dotenv()

llm = ChatOpenAI(
    api_key=os.getenv("API_KEY"),   # 直接读取.env的API_KEY
    base_url=os.getenv("BASE_URL"), # 直接读取.env的BASE_URL
    model=os.getenv("MODEL"),       # 直接读取.env的MODEL，这个也可以在.env中配置
    temperature=0.3
)

# =========================
# 3. 组件1：提取核心卖点
# =========================
sell_point_prompt = PromptTemplate(
    input_variables=["product_intro"],
    template="请从以下产品介绍中提取3个核心卖点，用简洁的语言列出：{product_intro}"
)

# 买点提取的 Runnable chain
sell_point_chain = sell_point_prompt | llm

# =========================
# 4. 中间结果结构化（LangChain 风格）
# =========================
extract_sell_points = RunnableLambda(
    lambda msg: {"sell_points": msg.content}
)

# =========================
# 5. 组件2：生成营销话术
# =========================
marketing_prompt = PromptTemplate(
    input_variables=["sell_points"],
    template="请根据以下产品核心卖点，写一段吸引消费者的营销话术（适合朋友圈发布）：{sell_points}"
)

# 营销话术生成的 Runnable chain
marketing_chain = marketing_prompt | llm

# =========================
# 6. 线性串联（Sequential Runnable）
# =========================
overall_chain = (
    sell_point_chain
    | extract_sell_points
    | marketing_chain
)

# =========================
# 7. 执行
# =========================
product_intro = """这款无线耳机采用蓝牙5.3芯片，连接稳定无延迟，支持高清通话；续航长达30小时，充电10分钟可使用2小时；机身采用亲肤硅胶材质，佩戴舒适，防水防汗，适合运动使用。"""

result = overall_chain.invoke({"product_intro": product_intro})

print("\n最终营销话术：")
print(result.content)


最终营销话术：
🔥【限时特惠】运动达人的必备神器来啦！🔥  

🎧 **蓝牙5.3超强芯**：告别卡顿断连！高清通话+无损音质，跑步听歌、开会通话，丝滑如风，稳到飞起！  

⚡ **30小时超长续航+闪充黑科技**：充电10分钟，畅听2小时！出差旅行不断电，运动一整天也电量满满！  

🏃‍♂️ **狂甩不掉的舒适感**：亲肤硅胶+人体工学设计，暴汗不滑落，防水防汗，戴着它，撸铁、瑜伽、登山……怎么动都稳如泰山！  

👉 现在下单立享粉丝专属价，点击秒杀↓  
#运动耳机 #蓝牙耳机 #高性价比神器  

（配图建议：产品佩戴场景图+充电动效图，突出运动感和科技感）


In [2]:
overall_chain

PromptTemplate(input_variables=['product_intro'], input_types={}, partial_variables={}, template='请从以下产品介绍中提取3个核心卖点，用简洁的语言列出：{product_intro}')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000029B45BB8410>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000029B460E2C10>, root_client=<openai.OpenAI object at 0x0000029B45B10F10>, root_async_client=<openai.AsyncOpenAI object at 0x0000029B460E1FD0>, model_name='deepseek-ai/DeepSeek-V3', temperature=0.3, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.siliconflow.cn/v1')
| RunnableLambda(lambda msg: {'sell_points': msg.content})
| PromptTemplate(input_variables=['sell_points'], input_types={}, partial_variables={}, template='请根据以下产品核心卖点，写一段吸引消费者的营销话术（适合朋友圈发布）：{sell_points}')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000029B45BB8410>, async_client=<openai.resources.ch

#### 4.1.2.3 进阶实践：多输入多输出复杂线性任务

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableMap, RunnablePassthrough
from dotenv import load_dotenv
import os

# 1. 初始化模型
load_dotenv()
llm = ChatOpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL"),
    model=os.getenv("MODEL"),
    temperature=0.3
)

# 2. Prompt 定义
sell_point_prompt = PromptTemplate(
    input_variables=["product_intro"],
    template="从以下产品介绍中提取3个核心卖点，简洁列出：{product_intro}"
)

marketing_prompt = PromptTemplate(
    input_variables=["sell_points", "target_audience"],
    template="针对{target_audience}，结合以下核心卖点，写一段朋友圈营销话术：{sell_points}"
)

# 3. 多输入多输出线性链（教学标准版）
overall_chain = (
    # Step 1：生成卖点 + 透传原始输入
    RunnableMap({
        "sell_points": sell_point_prompt | llm | (lambda x: x.content),
        "target_audience": RunnablePassthrough(),
    })
    # Step 2：营销话术生成
    | marketing_prompt
    | llm
)

# 4. 执行
input_data = {
    "product_intro": "这款无线耳机采用蓝牙5.3芯片，连接稳定无延迟...",
    "target_audience": "大学生群体（喜欢运动、预算有限、注重性价比）"
}

result = overall_chain.invoke(input_data)

print("营销话术：")
print(result.content)

营销话术：
【运动党&学生党必入！高性价比无线耳机来啦🎧】  

🔥蓝牙5.3芯片——运动不掉线！跑步/健身时狂甩也不断连，游戏追剧0延迟，稳到飞起~  

⚡30小时超长续航！图书馆刷题一整天+夜跑歌单循环，一周只充一次电，宿舍限电也不慌！  

🎯主动降噪黑科技！自习室键盘声/操场喧闹一键屏蔽，沉浸式学习&运动，百元价位千元体验！  

学生价XXX元立省一半！戳下方链接抢限量福利👇 #学生数码好物 #平价耳机王者  

（配图建议：耳机+运动场景/图书馆场景对比图）


【扩展：langchain_core.runnables模块概述】

`langchain_core.runnables` 是 LangChain 的核心模块，所有组件都继承自 `Runnable` 基类。

#### 核心类一览

| 类 | 说明 |
|---|---|
| **RunnableSequence** | 按顺序执行，`|` 运算符串联，上一输出作下一输入 |
| **RunnableParallel**（RunnableMap） | 并行执行多个组件，返回 dict |
| **RunnableBranch** | 根据条件选择不同分支 |
| **RunnableLambda** | 将 Python 函数转换为 Runnable |
| **RunnableGenerator** | 将生成器函数转换为 Runnable（流式处理） |
| **RunnablePassthrough** | 透传输入，可附加额外字段 |
| **RunnableAssign** | 为字典输入赋值 |
| **RunnablePick** | 从字典输入中选取指定 key |
| **RunnableBinding** | 为 Runnable 绑定固定配置 |
| **RunnableWithFallbacks** | 失败时切换到备用 Runnable |
| **RunnableWithMessageHistory** | 管理对话历史 |

#### 核心接口

`.invoke()`（同步执行）、`.stream()`（流式）、`.batch()`（批量）

#### 快速创建方式

- `chain = prompt | llm | output_parser`（`|` 运算符）
- `RunnableLambda(lambda x: x.upper())`（函数封装）


#### 4.2.2.1 多场景需求的动态路由匹配（如客服咨询分类处理）

In [4]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnableSequence
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
import os

# 1. 加载环境变量与初始化模型（新版推荐用ChatOpenAI，支持聊天模型）
load_dotenv()
llm = ChatOpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL"),
    model=os.getenv("MODEL"),
    temperature=0.3
)

# 2. 定义各场景的提示词模板与目标链（新版用RunnableSequence组合Prompt+LLM+Parser）
# 场景1：查订单链
order_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是智能客服，负责解答用户的订单查询问题。"),
    ("human", "用户问题：{query}\n请引导用户提供订单号，并告知查询流程：1. 提供订单号；2. 系统验证；3. 反馈订单状态。")
])
order_chain = order_prompt | llm | StrOutputParser()  # 新版Runnable流水线写法

# 场景2：退货款链
refund_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是智能客服，负责解答用户的退货款问题。"),
    ("human", "用户问题：{query}\n请说明退款流程：1. 申请退款（订单页面点击退款）；2. 等待审核（1-3个工作日）；3. 退款到账（原路返回，3-5个工作日）。如果用户问退款进度，引导提供退款申请单号。")
])
refund_chain = refund_prompt | llm | StrOutputParser()

# 场景3：保修政策链
warranty_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是智能客服，负责解答产品保修政策问题。"),
    ("human", "用户问题：{query}\n请说明保修政策：本产品保修期限为1年，保修范围包括质量问题（非人为损坏），保修流程：1. 联系客服；2. 提供购买凭证；3. 寄回检测维修。")
])
warranty_chain = warranty_prompt | llm | StrOutputParser()

# 3. 定义路由判断逻辑（大模型解析需求，输出场景标识）
# 路由提示词：让大模型输出标准化的场景名称，用于后续分支匹配
router_prompt = ChatPromptTemplate.from_messages([
    ("system", """
你是路由选择器，需根据用户问题判断所属场景，仅输出以下标准化标识之一：
- order：订单查询相关（含订单状态、订单号）
- refund：退货款相关（含退款进度、退款申请）
- warranty：保修相关（含维修、售后保障）
- default：以上均不匹配
无需输出任何其他内容，仅返回标识字符串。
"""),
    ("human", "用户问题：{query}")
])

# 路由解析链：输入query，输出场景标识
router_chain = router_prompt | llm | StrOutputParser()

# 4. 定义默认链（兜底处理）
default_prompt = PromptTemplate(
    input_variables=["query"],
    template="抱歉，我无法解答你的问题'{query}'。请你重新描述问题，或者联系人工客服（工作时间：9:00-18:00）。"
)
default_chain = default_prompt | llm | StrOutputParser()

# 5. 构建完整路由链（核心：RunnableBranch实现条件分发）
# 逻辑：先通过router_chain获取场景标识，再由RunnableBranch分发到对应目标链
full_router_chain = RunnableLambda(lambda x: x) | (
    # 分支1：匹配order场景
    RunnableBranch(
        (lambda x: x["scene"] == "order", order_chain),
        (lambda x: x["scene"] == "refund", refund_chain),
        (lambda x: x["scene"] == "warranty", warranty_chain),
        default_chain  # 默认分支
    )
).with_config(run_name="full_router_chain")


# 6. 封装调用函数（整合场景解析与路由分发）
def process_query(query: str):
    # 第一步：获取场景标识
    scene = router_chain.invoke({"query": query})
    # 第二步：将query和scene传入完整路由链，执行分发处理
    return full_router_chain.invoke({"query": query, "scene": scene})

# 7. 测试不同场景输入
test_queries = [
    "我的订单什么时候发货？",
    "怎么申请退款呀？",
    "这个产品保修多久？",
    "你们家有什么新品？"  # 无法匹配，触发默认链
]

for query in test_queries:
    print(f"\n用户问题：{query}")
    print("客服回复：", process_query(query))


用户问题：我的订单什么时候发货？
客服回复： 您好！为了帮您查询订单的发货时间，请您提供订单号。查询流程如下：
1. 请提供您的订单号
2. 系统会立即验证订单信息
3. 我将为您反馈最新的订单状态和预计发货时间

请问您的订单号是多少呢？

用户问题：怎么申请退款呀？
客服回复： 您好！申请退款的流程如下：

1. 申请退款：请您在订单页面找到需要退款的订单，点击"申请退款"按钮
2. 等待审核：提交后我们的客服会在1-3个工作日内审核您的申请
3. 退款到账：审核通过后，退款将在3-5个工作日内原路返回到您的支付账户

如果您需要查询退款进度，请提供您的退款申请单号，我可以帮您查看具体情况哦~

请问您是需要办理退款还是查询退款进度呢？

用户问题：这个产品保修多久？
客服回复： 您好！关于本产品的保修政策如下：

1. 保修期限：自购买之日起1年
2. 保修范围：产品质量问题（非人为损坏）
3. 保修流程：
   - 第一步：联系我们的客服热线或在线客服
   - 第二步：提供有效的购买凭证（如发票或订单信息）
   - 第三步：按客服指引寄回产品检测维修

温馨提示：保修服务不包含人为损坏或意外损坏的情况。如有其他疑问，欢迎随时咨询。

用户问题：你们家有什么新品？
客服回复： 感谢你的理解！如果你是想询问我们公司或品牌的新产品信息，建议你可以这样获取最新资讯：  

1. **访问官网**：我们官网的「新品发布」或「产品中心」版块会实时更新。  
2. **关注官方社交账号**：微信公众号、微博等平台会推送新品动态。  
3. **联系人工客服**：直接提供具体品牌或公司名称，我可以帮你查找官方渠道。  

如果你能提供更多细节（如具体行业、品牌名称等），我会尽力为你定向检索信息！ 🌟


#### 4.2.3.2 工程化解决方案：重试机制、异常捕获与分支降级

##### 1. 重试机制（解决临时API错误）

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.exceptions import OutputParserException
from langchain_core.runnables import Runnable
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()
llm = ChatOpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL"),
    model=os.getenv("MODEL"),
    temperature=0.3
)

# 1️⃣ Prompt
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "请简洁总结以下文本核心内容，不超过50字。"),
    ("human", "{text}")
])

# 2️⃣ 基础链
base_chain: Runnable = summary_prompt | llm | StrOutputParser()

# 3️⃣ 重试链（官方推荐：直接 with_retry）
retry_chain = base_chain.with_retry(
    stop_after_attempt=3,          # 最多重试 3 次
    wait_exponential_jitter=True,  # 指数退避 + 抖动（推荐）
    retry_if_exception_type=(
        ConnectionError, 
        TimeoutError,
    ),
)

# 4️⃣ 调用
try:
    result = retry_chain.invoke({
        "text": "LangChain是一个用于构建大模型应用的框架，提供了丰富的Runnable组件，支持重试、降级等工程化能力。"
    })
    print("总结结果：", result)

except OutputParserException as e:
    # ❗ 解析错误通常是逻辑问题，不建议重试
    print("输出解析失败：", e)

except Exception as e:
    print("最终失败（已达到最大重试次数）：", e)

总结结果： LangChain是构建大模型应用的框架，提供Runnable组件和工程化能力。


##### 2. 异常捕获（解决可预知的错误）

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.exceptions import OutputParserException
from langchain_core.runnables import Runnable
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()
llm = ChatOpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL"),
    model=os.getenv("MODEL"),
    temperature=0.3
)

In [7]:
# 1️⃣ 定义多变量 Prompt 链（营销话术生成示例）
marketing_prompt = ChatPromptTemplate.from_messages([
    ("system", "根据产品卖点和目标人群，撰写一句营销话术。"),
    ("human", "产品卖点：{sell_points}，目标人群：{target_audience}")
])

# 2️⃣ 构建链
marketing_chain: Runnable = marketing_prompt | llm | StrOutputParser()

# 3️⃣ 调用并捕获异常（官方推荐风格）
inputs = {
    "sell_points": "无线耳机续航30小时",
    # "target_audience" 故意缺失，用于演示 KeyError
}

try:
    result = marketing_chain.invoke(inputs)
    print("营销话术：", result)

except KeyError as e:
    # ✅ 实际抛出的是 KeyError，直接提取缺失的变量名
    missing_var = str(e).strip("'\"")
    print(f"错误提示：缺少必要输入变量 [{missing_var}]，请检查输入数据是否完整。")
except OutputParserException as e:
    # ✅ 官方推荐：逻辑解析错误不重试
    print(f"解析失败：{e}，请确认 Prompt 与输出格式匹配。")

except Exception as e:
    # ❗ 兜底捕获未知异常
    print(f"未知错误：{type(e).__name__}: {e}，请联系开发者排查。")

错误提示：缺少必要输入变量 [Input to ChatPromptTemplate is missing variables {'target_audience'}.  Expected: ['sell_points', 'target_audience'] Received: ['sell_points']\nNote: if you intended {target_audience} to be part of the string and not a variable, please escape it with double curly braces like: '{{target_audience}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT ]，请检查输入数据是否完整。


##### 3. 分支降级（错误时切换备用方案）

这里为了演示，调用的时候先传入openai的apikey，但其实.env里面没有正确的openai_apikey，所以会抛出连接异常，在捕获到异常后，切换到降级链，使用DeepSeek的api

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableWithFallbacks
from langchain_openai import ChatOpenAI
from langchain_core.exceptions import OutputParserException
import os
from dotenv import load_dotenv

load_dotenv()

# 🔑 使用 OpenAI 的 API 密钥（从环境变量读取）
openai_api_key = os.getenv("OPENAI_API_KEY")  # 在 .env 中设置 OPENAI_API_KEY=sk-xxxxx
# 注意：不需要指定 base_url，默认就是 https://api.openai.com

api_key = os.getenv("API_KEY")
base_url = os.getenv("BASE_URL")
model = os.getenv("MODEL")


# 1️⃣ 核心链（性能高但可能不稳定）
core_llm = ChatOpenAI(api_key=openai_api_key, model="gpt-4", temperature=0.7)
core_prompt = ChatPromptTemplate.from_messages(
    [("system", "用专业语言详细解答用户问题。"), ("human", "{query}")]
)
core_chain = core_prompt | core_llm | StrOutputParser()

# 2️⃣ 降级链（稳定但精度略低）
fallback_llm = ChatOpenAI(
    api_key=api_key, base_url=base_url, model=model, temperature=0.3
)
fallback_prompt = ChatPromptTemplate.from_messages(
    [("system", "用简洁语言解答用户问题，保证信息准确。"), ("human", "{query}")]
)
fallback_chain = fallback_prompt | fallback_llm | StrOutputParser()

# 3️⃣ 构建带降级的链
chain_with_fallback: RunnableWithFallbacks = core_chain.with_fallbacks(
    fallbacks=[fallback_chain],
    # exceptions_to_handle=(ConnectionError, TimeoutError),
    # 可以捕获 连接异常 和 超时异常
    # 这里为了示范就捕获所有异常，因为错误的apikey返回的应该是 认证异常
)

# 4️⃣ 调用链并捕获异常
try:
    result = chain_with_fallback.invoke({"query": "什么是RAG技术？"})
    print("解答：", result)
except OutputParserException as e:
    print(f"解析失败：{e}")
except Exception as e:
    print(f"最终失败：{e}")

# 最终调用成功，说明降级链正常工作

解答： RAG（Retrieval-Augmented Generation，检索增强生成）是一种结合信息检索与文本生成的技术，其核心流程分为两步：

1. **检索阶段**：当接收到用户问题后，系统从外部知识库（如数据库、文档集）中快速检索相关片段；
2. **生成阶段**：将检索到的信息与问题一起输入生成模型（如GPT），生成准确且依据可靠的回答。

主要优势：
- 提升回答准确性（基于实时检索而非仅依赖模型记忆）
- 可动态更新知识（仅需更新检索库，无需重新训练模型）
- 自带来源追溯（可标注引用文献）

典型应用场景：客服系统、医疗问答、法律咨询等需要精准专业知识的领域。例如，当用户询问最新政策时，RAG会先检索政府官网最新文件，再生成回答。


In [9]:
# 确定异常类型

try:
    result = core_chain.invoke({"query": "什么是RAG技术？"})
except Exception as e:
    print(f"发生异常: {type(e).__name__}")

发生异常: AuthenticationError


**【扩展】Langchain中常见异常类**

LangChain 的异常主要分布在 `langchain_core.exceptions` 和 `langchain_openai` 等模块中：

##### 1️⃣ 输入/输出相关

| 异常类 | 触发场景 | 示例 |
|--------|----------|------|
| **`KeyError`** | Prompt 缺少必需变量 | 调用时未传入 `{target_audience}` |
| **`InvalidPromptInputError`** | Prompt 输入格式错误 | 传入了非法的 prompt 变量 |
| **`OutputParserException`** | 输出解析失败 | LLM 输出无法按预期格式解析 |

##### 2️⃣ API/网络相关

| 异常类 | 触发场景 | 示例 |
|--------|----------|------|
| **`AuthenticationError`** | API 密钥错误或无效 | `401 Incorrect API key` |
| **`RateLimitError`** | 请求频率超限 | 短时间内请求过多 |
| **`TimeoutError`** | 请求超时 | 网络延迟或服务响应慢 |
| **`ConnectionError`** | 网络连接失败 | 无法连接到 API 服务器 |

##### 3️⃣ Chain/执行相关

| 异常类 | 触发场景 | 示例 |
|--------|----------|------|
| **`ChainDecodeError`** | Chain 解码/反序列化失败 | 加载保存的 Chain 配置出错 |
| **`MissingRuntimeArgsError`** | Runnable 运行时缺少参数 | 配置与实际调用不匹配 |

##### 4️⃣ 捕获建议

```python
# ✅ 推荐：按异常类型分别处理
try:
    result = chain.invoke(inputs)
except KeyError as e:
    # 输入变量缺失 → 检查输入数据
    print(f"缺少变量: {e}")
except OutputParserException as e:
    # 输出解析失败 → 不重试，检查 Prompt/Parser
    print(f"解析失败: {e}")
except (ConnectionError, TimeoutError) as e:
    # 网络问题 → 可以重试或降级
    print(f"网络错误: {e}")

# ⚠️ 注意：AuthenticationError 通常不推荐自动降级
# 错误的 API Key 意味着配置问题，降级到备用链也无法解决
```

##### 5️⃣ 快速获取异常类型

```python
# 不确定是什么异常？先打印类型
try:
    result = chain.invoke(inputs)
except Exception as e:
    print(f"异常类型: {type(e).__name__}")           # 如 "KeyError"
    print(f"异常模块: {type(e).__module__}")         # 如 "langchain_core.exceptions"
    print(f"异常信息: {e}")
```

#### 4.4.1.2 实操：多格式文档加载代码实现

##### TXT文档加载

In [10]:
# 导入TXT加载器（新版路径：langchain_community.document_loaders）
from langchain_community.document_loaders import TextLoader
import os

# 定义文档路径（请替换成你自己的路径）
txt_path = os.path.join("knowledge_base", "test.txt")

# 初始化加载器并加载文档
loader = TextLoader(txt_path, encoding="utf-8")  # 指定编码，避免中文乱码
txt_docs = loader.load()  # load()返回Document对象列表（即使只有一个文档）

# 查看加载结果
print("TXT文档加载结果：")
print(f"文档数量：{len(txt_docs)}")
print(f"文档内容：{txt_docs[0].page_content[:200]}...")  # 打印前200个字符
print(f"文档元数据：{txt_docs[0].metadata}")  # 元数据包含文档路径等信息

TXT文档加载结果：
文档数量：1
文档内容：LangChain 是一个用于构建大语言模型（LLM）应用程序的开源框架。它提供了丰富的组件，包括模型集成、提示词管理、记忆模块、工具调用和链式调用等功能。LangChain 支持 Python 和 JavaScript 两种语言，广泛应用于聊天机器人、文档问答、数据提取和自动化工作流等场景，帮助开发者快速构建智能应用。...
文档元数据：{'source': 'knowledge_base\\test.txt'}


##### PDF文档加载

1. 基础款（官方推荐，仅提取文本，轻量）

In [11]:
from langchain_community.document_loaders import PyPDFLoader
import os

# 定义PDF路径
pdf_path = os.path.join("knowledge_base", "test.pdf")

# 初始化加载器并加载（按页拆分）
loader = PyPDFLoader(pdf_path)
pdf_docs = loader.load_and_split()  # load_and_split()直接按页拆分，更易用
# 返回的是一个 list[Document] 文档对象列表
print(pdf_docs)

# 查看结果
print("\nPDF文档加载结果（基础款）：")
print(f"PDF总页数：{len(pdf_docs)}")
print(f"第1页内容：{pdf_docs[0].page_content[:200]}...")
print(f"第1页元数据：{pdf_docs[0].metadata}")  # 元数据包含页码、文档路径

[Document(metadata={'producer': '', 'creator': 'WPS 文字', 'creationdate': '2026-03-27T14:51:21+08:00', 'author': '25588', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-03-27T14:51:21+08:00', 'sourcemodified': "D:20260327145121+08'00'", 'subject': '', 'title': '', 'trapped': '/False', 'source': 'knowledge_base\\test.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='LangChain 框架介绍\nLangChain 是一个用于构建大语言模型（LLM）应用程序的开源框架。它提供了丰富的组件，\n包括模型集成、提示词管理、记忆模块、工具调用和链式调用等功能。LangChain 支持 Python\n和 JavaScript 两种语言，广泛应用于聊天机器人、文档问答、数据提取和自动化工作流等场\n景，帮助开发者快速构建智能应用。')]

PDF文档加载结果（基础款）：
PDF总页数：1
第1页内容：LangChain 框架介绍
LangChain 是一个用于构建大语言模型（LLM）应用程序的开源框架。它提供了丰富的组件，
包括模型集成、提示词管理、记忆模块、工具调用和链式调用等功能。LangChain 支持 Python
和 JavaScript 两种语言，广泛应用于聊天机器人、文档问答、数据提取和自动化工作流等场
景，帮助开发者快速构建智能应用。...
第1页元数据：{'producer': '', 'creator': 'WPS 文字', 'creationdate': '2026-03-27T14:51:21+08:00', 'author': '25588', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-03-27T14:51:21+08:00', '

2. 复杂款（需保留表格/格式时用）

In [13]:
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader(pdf_path)
pdf_docs_adv = loader.load()
print("\nPDF文档加载结果（复杂款）：")
print(f"第1页表格/格式保留情况：{pdf_docs_adv[0].page_content[:200]}...")


PDF文档加载结果（复杂款）：
第1页表格/格式保留情况：框架介绍
LangChain
LangChain 是一个用于构建大语言模型（LLM）应用程序的开源框架。它提供了丰富的组件，
包括模型集成、提示词管理、记忆模块、工具调用和链式调用等功能。LangChain 支持 Python
和 JavaScript 两种语言，广泛应用于聊天机器人、文档问答、数据提取和自动化工作流等场
景，帮助开发者快速构建智能应用。
...


【总结一下】

- PyPDFLoader 只提取文字
- PDFPlumberLoader 可以保留复杂的表格和格式